# 01. Temporal Knowledge Graph & Dataset Exploration

This notebook provides an in-depth exploratory data analysis of the datasets driving the **TKG-Verify** framework:
1. **ICEWS14** (Integrated Crisis Early Warning System 2014): Daily event-based temporal knowledge graph quadruples.
2. **CronQuestions**: Complex temporal question answering benchmark over Wikidata quintuples.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import json

# Ensure project root is in sys.path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from config import settings
from src.data_loader.icews_loader import ICEWS14Loader
from src.data_loader.cronquestions_loader import CronQuestionsLoader

## 1. Exploring ICEWS14 Dataset

The ICEWS14 dataset contains daily international political events from the year 2014 formatted as temporal quadruples:
`$(e_s, r, e_o, t)$` where `t` is a discrete daily timestamp.

In [ ]:
icews_loader = ICEWS14Loader(data_dir=settings.ICEWS14_DIR)
icews_stats = icews_loader.get_statistics()
print("ICEWS14 Statistics:")
for k, v in icews_stats.items():
    print(f"  {k}: {v}")

In [ ]:
splits = icews_loader.load_all()
train_facts = splits.get("train", [])
print(f"Loaded {len(train_facts):,} training facts.")
df_sample = pd.DataFrame(train_facts[:10])
df_sample

### Temporal Event Distribution across 2014

In [ ]:
dates = [f["timestamp"] for f in train_facts]
df_dates = pd.DataFrame({"timestamp": pd.to_datetime(dates)})
monthly_counts = df_dates["timestamp"].dt.to_period("M").value_counts().sort_index()

plt.figure(figsize=(10, 4))
monthly_counts.plot(kind="bar", color="skyblue", edgecolor="black")
plt.title("ICEWS14 Event Frequency by Month (2014)")
plt.xlabel("Month")
plt.ylabel("Number of Recorded Events")
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## 2. Exploring CronQuestions Dataset

CronQuestions evaluates temporal reasoning over Wikidata quintuples:
`$(e_s, r, e_o, t_{start}, t_{end})$` with corresponding natural language questions across multiple reasoning types (time-to-entity, entity-to-time, before/after, simultaneous).

In [ ]:
cron_loader = CronQuestionsLoader(data_dir=settings.CRONQUESTIONS_DIR)
cron_stats = cron_loader.get_statistics()
print("CronQuestions Statistics:")
for k, v in cron_stats.items():
    print(f"  {k}: {v}")

In [ ]:
test_questions = cron_loader.load_questions(split="test")
print(f"Sample test questions ({len(test_questions)} total):")
for q in test_questions[:5]:
    print(f"Q: {q['question']}")
    print(f"   Type: {q['answer_type']} | Expected: {q['answer']}")